In [5]:
import sqlite3

# Connexion à la base
conn = sqlite3.connect("immoprojetdefdef.db")
cursor = conn.cursor()
print("Connexion réussie à SQLite ✅")

Connexion réussie à SQLite ✅


In [3]:
import pandas as pd

df_region = pd.read_excel("fr-esr-referentiel-geographique.xlsx")

# Affiche les 5 premières lignes
df_region.head()


,regrgp_nom,reg_nom,reg_nom_old,aca_nom,dep_nom,com_code,com_code1,com_code2,com_id,com_nom_maj_court,...,fd_id,fr_id,fe_id,uu_id_99,au_code,au_id,auc_id,auc_nom,uu_id_10,geolocalisation
0,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01001,1001,1001,C01001,L ABERGEMENT CLEMENCIAT,...,FD111,FR11,FE1,SO,NaN,AU997,C01001,L'Abergement-Clémenciat,SO,"46.1534255214,4.92611354223"
1,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01002,1002,1002,C01002,L ABERGEMENT DE VAREY,...,FD111,FR11,FE1,SO,2,AU002,AU002,Lyon,SO,"46.0091878776,5.42801696363"
2,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01003,1003,1003,C01003,AMAREINS,...,FD111,FR11,FE1,SO,SO,SO,SO,SO,SO,NaN
3,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01004,1004,1004,C01004,AMBERIEU EN BUGEY,...,FD111,FR11,FE1,UU01303,2,AU002,AU002,Lyon,UU01302,"45.9608475114,5.3729257777"
4,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01005,1005,1005,C01005,AMBERIEUX EN DOMBES,...,FD111,FR11,FE1,SO,2,AU002,AU002,Lyon,SO,"45.9961799872,4.91227250796"


In [7]:
# Renommer les colonnes
df_region = df_region.rename(columns={
    "reg_code": "code_region",
    "reg_nom": "nom_region"
})

# Ne garder que les colonnes nécessaires
df_region = df_region[["code_region", "nom_region"]]

# Supprimer les doublons
df_region = df_region.drop_duplicates(subset=["code_region"])

# Insérer dans la base
df_region.to_sql("Region", conn, if_exists="append", index=False)

print("✅ Données insérées dans la table Region")

# Vérifier les 5 premières lignes
result = cursor.execute("SELECT * FROM Region LIMIT 5").fetchall()
print("\n🧾 Premières lignes de la table Region :")
for row in result:
    print(row)


✅ Données insérées dans la table Region

🧾 Premières lignes de la table Region :
('84', 'Auvergne-Rhône-Alpes')
('32', 'Hauts-de-France')
('93', "Provence-Alpes-Côte d'Azur")
('44', 'Grand Est')
('76', 'Occitanie')


In [8]:


# Charger le fichier Excel
df_geo = pd.read_excel("fr-esr-referentiel-geographique.xlsx")

# Créer le DataFrame pour Departement
df_departement = df_geo[["dep_code", "dep_nom", "reg_code"]].drop_duplicates()

# Renommer les colonnes pour correspondre à la table SQL
df_departement = df_departement.rename(columns={
    "dep_code": "code_departement",
    "dep_nom": "nom_departement",
    "reg_code": "code_region"
})

# Vérification des doublons sur la PK
df_departement = df_departement.drop_duplicates(subset=["code_departement"])

# Insérer dans la table Departement
df_departement.to_sql("Departement", conn, if_exists="append", index=False)

print("✅ Données insérées dans la table Departement")

# Vérifier les 5 premières lignes
result = cursor.execute("SELECT * FROM Departement LIMIT 5").fetchall()
print("\n🧾 Premiers départements insérés :")
for row in result:
    print(row)


✅ Données insérées dans la table Departement

🧾 Premiers départements insérés :
('1', 'Ain', '84')
('2', 'Aisne', '32')
('3', 'Allier', '84')
('4', 'Alpes-de-Haute-Provence', '93')
('5', 'Hautes-Alpes', '93')


In [11]:


# --------------------------------------------------------
# 1. Charger Valeurs-foncières.xlsx (informations communes)
# --------------------------------------------------------
vf = pd.read_excel("Valeurs-foncières.xlsx")

# Garder uniquement les colonnes utiles
df_vf = vf[["Code departement", "Code commune", "Commune", "Code postal"]].dropna()

# Renommer pour correspondre à la base
df_vf = df_vf.rename(columns={
    "Code departement": "code_departement",
    "Code commune": "code_commune",
    "Commune": "nom_commune",
    "Code postal": "code_postal"
})

# Uniformiser types
df_vf["code_departement"] = df_vf["code_departement"].astype(str).str.zfill(2)
df_vf["code_commune"] = df_vf["code_commune"].astype(int)

# Supprimer doublons
df_vf = df_vf.drop_duplicates(subset=["code_departement", "code_commune"])

# --------------------------------------------------------
# 2. Charger donnees_communes.xlsx (population par commune)
# --------------------------------------------------------
demo = pd.read_excel("donnees_communes.xlsx")

# Garder les colonnes utiles
df_demo = demo[["CODDEP", "CODCOM", "PTOT"]]

# Renommer pour correspondre
df_demo = df_demo.rename(columns={
    "CODDEP": "code_departement",
    "CODCOM": "code_commune",
    "PTOT": "population"
})

df_demo["code_departement"] = df_demo["code_departement"].astype(str).str.zfill(2)
df_demo["code_commune"] = df_demo["code_commune"].astype(int)

# --------------------------------------------------------
# 3. Fusion des deux jeux de données
# --------------------------------------------------------
df_commune = pd.merge(df_vf, df_demo, on=["code_departement", "code_commune"], how="left")

# --------------------------------------------------------
# 4. Création de la clé primaire id_coddep_codecommune
# --------------------------------------------------------
df_commune["id_coddep_codecommune"] = df_commune["code_departement"] + df_commune["code_commune"].apply(lambda x: f"{x:03d}")

# Réorganisation des colonnes pour insertion
df_commune = df_commune[[
    "id_coddep_codecommune",
    "code_departement",
    "code_commune",
    "code_postal",
    "nom_commune",
    "population"
]]

# Supprimer doublons éventuels
df_commune = df_commune.drop_duplicates(subset=["id_coddep_codecommune"])

# --------------------------------------------------------
# 5. Affichage de 10 lignes AVANT INSERTION
# --------------------------------------------------------
print("🧾 Extrait des données à insérer dans la table Commune :")
print(df_commune.head(10))

# --------------------------------------------------------
# 6. Insertion dans la base
# --------------------------------------------------------
df_commune.to_sql("Commune", conn, if_exists="append", index=False)
print("\n✅ Insertion terminée dans la table Commune.")


🧾 Extrait des données à insérer dans la table Commune :
  id_coddep_codecommune code_departement  code_commune  code_postal  \
0                 01103               01           103       1170.0   
1                 06004               06             4       6160.0   
2                 06088               06            88       6000.0   
3                 06123               06           123       6700.0   
4                 13005               13             5      13400.0   
5                 13028               13            28      13600.0   
6                 13208               13           208      13008.0   
7                 13212               13           212      13012.0   
8                 14338               14           338      14510.0   
9                 14366               14           366      14100.0   

            nom_commune  population  
0                CHEVRY        2196  
1               ANTIBES       74523  
2                  NICE      345528  
3  SAINT L

In [12]:
# --------------------------------------------------------
# 1. Charger Valeurs-foncières.xlsx
# --------------------------------------------------------
vf = pd.read_excel("Valeurs-foncières.xlsx")

# Vérifier que les colonnes existent
colonnes_utiles = [
    "Code departement", "Code commune", "No voie", "B/T/Q", "Type de voie",
    "Voie", "Nombre pieces principales", "Surface Carrez du 1er lot",
    "Surface reelle bati", "Type local"
]

# Sélectionner les colonnes
df_bien = vf[colonnes_utiles].copy()

# Renommer les colonnes pour correspondre à la table Bien
df_bien = df_bien.rename(columns={
    "Code departement": "code_departement",
    "Code commune": "code_commune",
    "No voie": "no_voie",
    "B/T/Q": "btq",
    "Type de voie": "type_voie",
    "Voie": "voie",
    "Nombre pieces principales": "total_piece",
    "Surface Carrez du 1er lot": "surface_carrez",
    "Surface reelle bati": "surface_local",
    "Type local": "type_local"
})

# Nettoyage
df_bien["code_departement"] = df_bien["code_departement"].astype(str).str.zfill(2)
df_bien["code_commune"] = df_bien["code_commune"].astype(int)

# Création de id_coddep_codecommune
df_bien["id_coddep_codecommune"] = df_bien["code_departement"] + df_bien["code_commune"].apply(lambda x: f"{x:03d}")

# Réorganisation des colonnes pour insertion
df_bien = df_bien[[
    "id_coddep_codecommune",
    "no_voie",
    "btq",
    "type_voie",
    "voie",
    "total_piece",
    "surface_carrez",
    "surface_local",
    "type_local"
]]

# --------------------------------------------------------
# 2. Aperçu des données AVANT insertion
# --------------------------------------------------------
print("🧾 Extrait des données à insérer dans la table Bien :")
print(df_bien.head(10))

# --------------------------------------------------------
# 3. Insertion dans la base
# --------------------------------------------------------
#df_bien.to_sql("Bien", conn, if_exists="append", index=False)
#print("\n✅ Insertion réussie dans la table Bien.")


🧾 Extrait des données à insérer dans la table Bien :
  id_coddep_codecommune  no_voie  btq type_voie                    voie  \
0                 01103    347.0  NaN       RUE              DU CHATEAU   
1                 06004      4.0  NaN        BD         EDOUARD BAUDOIN   
2                 06088     20.0    B       RUE                 MARCEAU   
3                 06123    550.0  NaN       RTE         DES VESPINS RN7   
4                 13005   9300.0  NaN       RES  LES ARPEGES BD DES ABA   
5                 13028     27.0  NaN       RUE         DU GRAND MADIER   
6                 13208    360.0  NaN        AV                DU PRADO   
7                 13212   5076.0    F      PARC                DESSUARD   
8                 14338   1194.0  NaN       RUE            DE NORMANDIE   
9                 14366     30.0  NaN       ALL          DES NOISETIERS   

   total_piece  surface_carrez  surface_local   type_local  
0            3           48.22             48  Appartement  

In [13]:
# --------------------------------------------------------
# 3. Insertion dans la base
# --------------------------------------------------------
df_bien.to_sql("Bien", conn, if_exists="append", index=False)
print("\n✅ Insertion réussie dans la table Bien.")



✅ Insertion réussie dans la table Bien.


In [14]:

# --------------------------------------------------------
# 1. Charger Valeurs-foncières.xlsx
# --------------------------------------------------------
vf = pd.read_excel("Valeurs-foncières.xlsx")

# Ne garder que les colonnes nécessaires
df_vente = vf[["Date mutation", "Valeur fonciere"]].copy()
df_vente = df_vente.rename(columns={
    "Date mutation": "date",
    "Valeur fonciere": "valeur"
})

# Nettoyer les dates (convertir en format YYYY-MM-DD)
df_vente["date"] = pd.to_datetime(df_vente["date"], errors="coerce").dt.strftime('%Y-%m-%d')

# Supprimer les lignes sans date
df_vente = df_vente.dropna(subset=["date"])

# Ajouter la clé étrangère id_bien
# Hypothèse : correspondance 1-1 avec ordre d'insertion de Bien
# On récupère les id_bien existants dans le même ordre

# Récupérer tous les id_bien de la base, dans l'ordre d'insertion
ids_bien = pd.read_sql_query("SELECT id_bien FROM Bien", conn)

# Ajouter à df_vente
df_vente = df_vente.reset_index(drop=True)
df_vente["id_bien"] = ids_bien["id_bien"]

# Réorganiser les colonnes pour insertion
df_vente = df_vente[["id_bien", "date", "valeur"]]

# --------------------------------------------------------
# 2. Affichage avant insertion
# --------------------------------------------------------
print("🧾 Extrait des données à insérer dans la table Vente :")
print(df_vente.head(100))



🧾 Extrait des données à insérer dans la table Vente :
    id_bien        date    valeur
0         1  2020-01-02  165000.0
1         2  2020-01-02  355680.0
2         3  2020-01-02  229500.0
3         4  2020-01-02  125000.0
4         5  2020-01-02   90000.0
..      ...         ...       ...
95       96  2020-01-03  142000.0
96       97  2020-01-03  164000.0
97       98  2020-01-03   33880.0
98       99  2020-01-03   71000.0
99      100  2020-01-03   43000.0

[100 rows x 3 columns]


In [15]:

# --------------------------------------------------------
# 3. Insertion dans la base
# --------------------------------------------------------
df_vente.to_sql("Vente", conn, if_exists="append", index=False)
print("\n✅ Insertion réussie dans la table Vente.")


✅ Insertion réussie dans la table Vente.


In [16]:
import os
print("Fichier base de données SQLite :", os.path.abspath("immoprojetdefdef.db"))


Fichier base de données SQLite : C:\Users\karap\OpenClassRooms\dataprojet3\immoprojetdefdef.db
